# Comparación de algoritmos de aprendizaje supervisado: KNN, SVM y red neuronal (MLP)

**Equipo:** CafeSmart **Integrantes:** Morales Mendez Shryley Nicoles, Jimenez Andrade Monserrat, Tejero Hernandez Rubi, Sarao Morales Rubi.

**Objetivo.** Implementar y comparar tres algoritmos (KNN, SVM y MLP) con el mismo conjunto de datos y la misma división de entrenamiento y prueba, para clasificar tumores de mama como **benignos** o **malignos** (dataset *Wisconsin Diagnostic Breast Cancer*, WDBC).

**Contenido:** 1. Carga y descripción de los datos · 2. Exploración · 3. Variables, división y preprocesamiento · 4. Entrenamiento y métricas · 5. Tabla comparativa y gráficas · 6. Ajuste de hiperparámetros (complemento) · 7. ¿Qué tan confiable es la comparación? · 8. Respuestas a las preguntas · 9. Conclusiones.

Todos los resultados salen de la ejecución de este notebook (semilla fija `SEED = 42`).

In [ ]:
# Instalación de librerías.
# Quita "#" de la línea siguiente, ejecuta la celda y vuelve a comentarla.
%pip install numpy pandas matplotlib scikit-learn jinja2

IndentationError: unexpected indent (2986614808.py, line 3)

1. Carga y descripción de los datos
El archivo wdbc.data no trae encabezados: tiene el identificador, el diagnóstico (M o B) y 30 características numéricas. Los nombres se tomaron de wdbc.names: 10 medidas del núcleo celular (radio, textura, perímetro, área, suavidad, compacidad, concavidad, puntos cóncavos, simetría y dimensión fractal), cada una como media, error estándar y peor valor (el promedio de los tres valores más grandes).

In [ ]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, RepeatedStratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)

SEED = 42
pd.set_option("display.max_columns", 40)

# --- Carga del archivo ---
URL_RAW = ""   # opcional: pega aquí la URL "raw" de wdbc.data de repositorio de GitHub
RUTA = next((r for r in ["wdbc.data", "data/wdbc.data"] if os.path.exists(r)), None)
if RUTA is None and URL_RAW:
    RUTA = URL_RAW
if RUTA is None:
    try:
        from google.colab import files
        print("Sube el archivo wdbc.data")
        files.upload()
        RUTA = "wdbc.data"
    except ImportError:
        raise FileNotFoundError("No se encontró wdbc.data: colócalo junto al notebook.")

base = ["radius", "texture", "perimeter", "area", "smoothness",
        "compactness", "concavity", "concave_points", "symmetry", "fractal_dimension"]
columnas = ["id", "diagnosis"] + [f"{b}_{s}" for s in ("mean", "se", "worst") for b in base]
df = pd.read_csv(RUTA, header=None, names=columnas)
df.head()